In [2]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)丨:

# ---------- 网络定义 ----------
class PINN(nn.Module):
    def __init__(self, layers=[2, 32, 32, 32, 1]):
        super().__init__()
        self.net = nn.Sequential()
        for i in range(len(layers)-1):
            self.net.append(nn.Linear(layers[i], layers[i+1]))
            if i < len(layers)-2:
                self.net.append(nn.Tanh())
    def forward(self, x):
        return self.net(x)

model = PINN()

# ---------- 采样点 ----------
N_interior = 4000
N_boundary = 800
# 内部点：均匀随机
x_int = torch.rand(N_interior, 2) * 2 - 1  # [-1,1]
# 边界点：每条边200个
x_bnd = []
# 使用 unsqueeze(0) 变成 [1, 1] 的形状，或者直接用 shape=(1,1)
x_bnd.append([torch.rand(1).unsqueeze(0)*2-1, torch.tensor([[-1.0]])])
x_bnd.append([torch.rand(1).unsqueeze(0)*2-1, torch.tensor([[1.0]])])
x_bnd.append([torch.tensor([[-1.0]]), torch.rand(1).unsqueeze(0)*2-1])
x_bnd.append([torch.tensor([[1.0]]), torch.rand(1).unsqueeze(0)*2-1])

# 此时内部 pair 里的张量都是 [1,1]，可以在 dim=1 拼接
x_bnd = torch.cat([torch.cat(pair, dim=1) for pair in x_bnd], dim=0)
# 解析解用于验证
x_val = torch.linspace(-1,1,101)
Xv, Yv = torch.meshgrid(x_val, x_val, indexing='ij')
x_test = torch.stack([Xv.ravel(), Yv.ravel()], dim=1)
u_exact = (torch.sin(np.pi*x_test[:,0]) * torch.sin(np.pi*x_test[:,1])).reshape(101,101)

# ---------- 损失函数 ----------
def pde_loss(model, x):
    x.requires_grad = True
    u = model(x)
    du = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u), create_graph=True)[0]
    u_xx = torch.autograd.grad(du[:,0], x, grad_outputs=torch.ones_like(du[:,0]), create_graph=True)[0][:,0]
    u_yy = torch.autograd.grad(du[:,1], x, grad_outputs=torch.ones_like(du[:,1]), create_graph=True)[0][:,1]
    residual = -(u_xx + u_yy) - (2*np.pi**2 * torch.sin(np.pi*x[:,0]) * torch.sin(np.pi*x[:,1]))
    return torch.mean(residual**2)

def bc_loss(model, x):
    u = model(x)
    return torch.mean(u**2)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# ---------- 训练 ----------
epochs = 2000
for ep in range(epochs):
    optimizer.zero_grad()
    loss = pde_loss(model, x_int) + 10.0 * bc_loss(model, x_bnd)  # 边界权重加大
    loss.backward()
    optimizer.step()
    if ep % 500 == 0:
        with torch.no_grad():
            u_pred = model(x_test).reshape(101,101)
            err = torch.abs(u_pred - u_exact).max().item()
        print(f'Epoch {ep:4d} | Loss {loss.item():.2e} | Max Error {err:.2e}')

Epoch    0 | Loss 9.79e+01 | Max Error 1.12e+00
Epoch  500 | Loss 1.70e-01 | Max Error 2.14e+00
Epoch 1000 | Loss 2.14e-02 | Max Error 1.50e+00
Epoch 1500 | Loss 8.86e-03 | Max Error 1.31e+00
